# 07 — Autoencoder (AE) e Variational Autoencoder (VAE) — senza feature displacement

**Prerequisito:** eseguire prima `preprocess_fin.ipynb` fino in fondo.
Questo notebook si aspetta il file `embeddings_preprocessed.npz` nella stessa cartella dei dati.

**Obiettivo:** identico al notebook `06`, ma con le feature esplicitamente progettate da ATLAS
per identificare tracce displaced **rimosse** dall'embedding, così da valutare quanto l'AUC
dipendesse da esse.

**Feature RIMOSSE (21 dim su 86):**

| Feature | Tipo | Dims |
|---------|------|------|
| `d0_log1p` | track (×5 stat) | 5–9 |
| `radiusOfFirstHit` | track (×5 stat) | 45–49 |
| `IP3D_signed_d0_significance` | track (×5 stat) | 50–54 |
| `IP3D_signed_z0_significance` | track (×5 stat) | 55–59 |
| `displacedPtFraction` | jet | 80 |

**Embedding risultante: 65 dim** (era 86)

| Sezione | Contenuto |
|---------|----------|
| 0 | Setup |
| 1 | Feature mask + caricamento dati |
| 2 | DataLoader PyTorch |
| 3 | MLP Autoencoder — architettura e training |
| 4 | Diagnostica AE |
| 5 | Variational Autoencoder (VAE) |
| 6 | Confronto AE vs VAE |
| 7 | Salvataggio |


## 0. Setup

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')

# ── Riproducibilità ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Percorsi ─────────────────────────────────────────────────────────────────
BASE_DIR   = Path(r'C:\Users\delco\Desktop\Universita\Ad_M\Dataset-Programs')
EMBED_PATH = BASE_DIR / 'outputs' / 'embeddings_preprocessed.npz'
OUT_DIR    = BASE_DIR / 'outputs' / 'ae_out_feat'   # separato da ae_out del 06
OUT_DIR.mkdir(exist_ok=True)

status = '✓ trovato' if EMBED_PATH.exists() else '✗ NON TROVATO — esegui prima preprocess_fin.ipynb'
print(f'  {EMBED_PATH.name}: {status}')

plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})


## 1. Feature mask + caricamento dati

Costruiamo la maschera sulle dimensioni dell'embedding:
- dims 0–74 : 5 statistiche (mean, std, p25, p75, max) × 15 track features
- dim 75    : n_valid_tracks normalizzato
- dims 76–85: 10 jet features

Vengono rimosse le feature legate al displacement (21 dim totali).


In [ ]:
# ── Definizione delle feature nell'embedding (dal preprocessing) ───────────────
TRACK_FEATURES = [
    'pt_log1p',                          #  0
    'd0_log1p',                          #  1  ← RIMOSSA
    'z0RelativeToBeamspot',              #  2
    'deta',                              #  3
    'dphi',                              #  4
    'dr',                                #  5
    'ptfrac',                            #  6
    'qOverP',                            #  7
    'chiSquared',                        #  8
    'radiusOfFirstHit',                  #  9  ← RIMOSSA
    'IP3D_signed_d0_significance',       # 10  ← RIMOSSA
    'IP3D_signed_z0_significance',       # 11  ← RIMOSSA
    'numberOfInnermostPixelLayerHits',   # 12
    'numberOfPixelHits',                 # 13
    'numberOfSCTHits',                   # 14
]
JET_FEATURES = [
    'pt',                          #  0  → dim 76
    'eta',                         #  1  → dim 77
    'mass',                        #  2  → dim 78
    'energy',                      #  3  → dim 79
    'displacedPtFraction',         #  4  → dim 80  ← RIMOSSA
    'Tau21_clusterSoftDrop',       #  5  → dim 81
    'Tau32_clusterSoftDrop',       #  6  → dim 82
    'Split12_clusterSoftDrop',     #  7  → dim 83
    'timing_clusterSoftDrop',      #  8  → dim 84
    'nPrimaryVertices',            #  9  → dim 85
]

REMOVE_TRACK = {'d0_log1p', 'radiusOfFirstHit',
                'IP3D_signed_d0_significance', 'IP3D_signed_z0_significance'}
REMOVE_JET   = {'displacedPtFraction'}

N_TF = len(TRACK_FEATURES)  # 15

keep_dims = []
for i, feat in enumerate(TRACK_FEATURES):
    base = 5 * i
    if feat not in REMOVE_TRACK:
        keep_dims.extend(range(base, base + 5))
    else:
        print(f'  REMOVED track feat [{i:2d}] {feat:35s} dims {base}–{base+4}')

keep_dims.append(5 * N_TF)   # dim 75: n_valid_tracks

for j, feat in enumerate(JET_FEATURES):
    dim = 5 * N_TF + 1 + j   # dims 76–85
    if feat not in REMOVE_JET:
        keep_dims.append(dim)
    else:
        print(f'  REMOVED jet  feat [{j:2d}] {feat:35s} dim  {dim}')

FEAT_MASK      = np.array(keep_dims, dtype=np.int64)
N_EMB          = len(FEAT_MASK)  # 65

print(f'\nEmbedding originale : 86 dim')
print(f'Feature rimosse     : {86 - N_EMB} dim')
print(f'Embedding ridotto   : {N_EMB} dim')


In [ ]:
def apply_mask(X: np.ndarray) -> np.ndarray:
    """Seleziona le dimensioni non-displacement: (N, 86) → (N, 65)."""
    return X[:, FEAT_MASK]

t0 = time.time()
data = np.load(EMBED_PATH, allow_pickle=True)

X_train = apply_mask(data['X_train'])   # (N_train, 65) float32
y_train = data['y_train']
X_val   = apply_mask(data['X_val'])     # (N_val,   65) float32
y_val   = data['y_val']
X_test  = apply_mask(data['X_test'])    # (N_test,  65) float32
y_test  = data['y_test']

print(f'Caricamento completato in {time.time()-t0:.2f}s')
print(f'N_EMB = {N_EMB}')
print()
for name, X, y in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    n_qcd = int((y == 0).sum())
    n_ej  = int((y == 1).sum())
    print(f'  {name:5s}: {len(X):6,} jet  (QCD={n_qcd:,}, EJ={n_ej:,})  shape={X.shape}')

# Il modello si addestra SOLO su QCD del training set
X_train_qcd = X_train[y_train == 0]
print(f'\nTraining set AE: {len(X_train_qcd):,} jet QCD (label=0)')


## 2. DataLoader PyTorch

In [ ]:
BATCH_SIZE = 1024

# Training: solo QCD
train_tensor = torch.from_numpy(X_train_qcd)

# Val QCD — usato per early stopping
X_val_qcd      = X_val[y_val == 0]
val_qcd_tensor = torch.from_numpy(X_val_qcd)

# Val completo QCD+EJ — solo diagnostica
val_tensor = torch.from_numpy(X_val)

loader_train   = DataLoader(TensorDataset(train_tensor),   batch_size=BATCH_SIZE, shuffle=True)
loader_val_qcd = DataLoader(TensorDataset(val_qcd_tensor), batch_size=BATCH_SIZE, shuffle=False)
loader_val     = DataLoader(TensorDataset(val_tensor),     batch_size=BATCH_SIZE, shuffle=False)

print(f'Training:  {len(train_tensor):,} QCD jet')
print(f'Val QCD:   {len(val_qcd_tensor):,} jet  (early stopping)')
print(f'Val mixed: {len(val_tensor):,} jet (QCD+EJ, solo diagnostica)')
print(f'Batch size: {BATCH_SIZE} | Steps/epoch: {len(loader_train)}')


## 3. MLP Autoencoder

### 3.1 Architettura

```
INPUT (65) → [65→64→32] → LATENT (16) → [16→32→64] → OUTPUT (65)
```

Identica al notebook 06, con `input_dim=65` invece di 86.


In [ ]:
class MLP_AE(nn.Module):
    """
    Autoencoder MLP per anomaly detection su jet.

    Parametri
    ---------
    input_dim  : dimensione embedding in input (65, senza feature displacement)
    latent_dim : dimensione spazio latente
    hidden     : lista dimensioni strati nascosti dell'encoder
    dropout    : dropout nel decoder (regolarizzazione leggera)
    """

    def __init__(self, input_dim=65, latent_dim=16, hidden=[64, 32], dropout=0.05):
        super().__init__()
        self.input_dim  = input_dim
        self.latent_dim = latent_dim

        # Encoder: input → ... → latent
        enc = []
        prev = input_dim
        for h in hidden:
            enc += [nn.Linear(prev, h), nn.LayerNorm(h), nn.GELU()]
            prev = h
        enc += [nn.Linear(prev, latent_dim)]
        self.encoder = nn.Sequential(*enc)

        # Decoder: latent → ... → input
        dec = []
        prev = latent_dim
        for h in reversed(hidden):
            dec += [nn.Linear(prev, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        dec += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*dec)

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def anomaly_score(self, x):
        """MSE per sample: shape (B,). Chiamare con torch.no_grad()."""
        reco = self.forward(x)
        return F.mse_loss(reco, x, reduction='none').mean(dim=1)


LATENT_DIM = 16

model_ae = MLP_AE(
    input_dim  = N_EMB,
    latent_dim = LATENT_DIM,
    hidden     = [64, 32],
    dropout    = 0.05,
).to(device)

n_params = sum(p.numel() for p in model_ae.parameters() if p.requires_grad)
print(model_ae)
print(f'\nParametri totali: {n_params:,}')


### 3.2 Training loop

In [ ]:
def train_ae(model, loader_train, loader_val_qcd, n_epochs=200, lr=3e-4, patience=20, verbose=True):
    """
    Training loop AE con early stopping su val QCD soltanto.

    Perché QCD-only per l'early stopping:
    il val set misto QCD+EJ ha una MSE enorme (gli EJ hanno feature molto
    fuori distribuzione dopo normalizzazione QCD) → la val loss mista è
    inutile per decidere quando fermarsi. Usiamo solo i QCD del val set.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=8
    )
    history = {'train': [], 'val_qcd': []}
    best_val, best_state, no_improve = float('inf'), None, 0

    for epoch in range(1, n_epochs + 1):
        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        tot = 0.
        for (xb,) in loader_train:
            xb = xb.to(device)
            optimizer.zero_grad()
            loss = F.mse_loss(model(xb), xb)
            loss.backward()
            optimizer.step()
            tot += loss.item() * len(xb)
        history['train'].append(tot / len(loader_train.dataset))

        # ── Val QCD (per early stopping) ──────────────────────────────────────
        model.eval()
        tot = 0.
        with torch.no_grad():
            for (xb,) in loader_val_qcd:
                xb = xb.to(device)
                tot += F.mse_loss(model(xb), xb).item() * len(xb)
        val_loss = tot / len(loader_val_qcd.dataset)
        history['val_qcd'].append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if verbose and (epoch % 20 == 0 or epoch == 1):
            print(f'Epoch {epoch:4d}/{n_epochs} | train={history["train"][-1]:.5f} | '
                  f'val_qcd={val_loss:.5f} | lr={optimizer.param_groups[0]["lr"]:.2e}')

        if no_improve >= patience:
            print(f'Early stopping a epoch {epoch}.')
            break

    model.load_state_dict(best_state)
    print(f'Best val QCD loss: {best_val:.5f}')
    return history


print('=== Training AE [no-disp] ===')
t0 = time.time()
history_ae = train_ae(model_ae, loader_train, loader_val_qcd, n_epochs=200, lr=3e-4, patience=20)
print(f'Tempo totale: {time.time()-t0:.1f}s')


In [ ]:
# ── Loss curve ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ep = range(1, len(history_ae['train']) + 1)
ax.plot(ep, history_ae['train'],   label='Train (QCD)')
ax.plot(ep, history_ae['val_qcd'], label='Val (QCD)', linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('AE [no-disp] — Loss curve')
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / 'ae_loss.png', dpi=120)
plt.show()


## 4. Diagnostica AE

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

def compute_scores(model, X_np, batch_size=4096):
    """Calcola l'anomaly score (MSE per jet) da un array numpy."""
    model.eval()
    scores = []
    with torch.no_grad():
        for i in range(0, len(X_np), batch_size):
            xb = torch.from_numpy(X_np[i:i+batch_size]).to(device)
            scores.append(model.anomaly_score(xb).cpu().numpy())
    return np.concatenate(scores)

# Anomaly scores sul test set
scores_ae = compute_scores(model_ae, X_test)
s_qcd_ae  = scores_ae[y_test == 0]
s_ej_ae   = scores_ae[y_test == 1]
auc_ae    = roc_auc_score(y_test, scores_ae)

print(f'AUC AE [no-disp]: {auc_ae:.4f}')
print(f'  QCD → mean={s_qcd_ae.mean():.4f}  P95={np.percentile(s_qcd_ae,95):.4f}')
print(f'  EJ  → mean={s_ej_ae.mean():.4f}   P05={np.percentile(s_ej_ae,5):.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Score distribution
ax = axes[0]
lo, hi = np.percentile(scores_ae, [0.5, 99.5])
bins = np.linspace(lo, hi, 80)
ax.hist(s_qcd_ae, bins=bins, density=True, alpha=0.6, label='QCD', color='steelblue')
ax.hist(s_ej_ae,  bins=bins, density=True, alpha=0.6, label='EJ',  color='tomato')
ax.set_yscale('log')
ax.set_xlabel('Anomaly score (MSE)')
ax.set_title(f'AE [no-disp] — Score distributions (test) | AUC={auc_ae:.4f}')
ax.legend()

# ROC
ax = axes[1]
fpr, tpr, _ = roc_curve(y_test, scores_ae)
ax.plot(fpr, tpr, color='steelblue', label=f'AE no-disp  AUC={auc_ae:.4f}')
ax.plot([0,1],[0,1],'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate (QCD mistag)')
ax.set_ylabel('True Positive Rate (EJ efficiency)')
ax.set_title('ROC — AE [no-disp]')
ax.legend()

fig.tight_layout()
fig.savefig(OUT_DIR / 'ae_scores_roc.png', dpi=120)
plt.show()


## 5. Variational Autoencoder (VAE)

In [ ]:
class MLP_VAE(nn.Module):
    """
    Variational Autoencoder MLP.

    Parametri
    ---------
    input_dim  : 65 (senza feature displacement)
    latent_dim : dimensione spazio latente
    hidden     : strati nascosti encoder
    beta       : peso del termine KL. beta=1 → VAE standard.
    """

    def __init__(self, input_dim=65, latent_dim=16, hidden=[64, 32], beta=1.0):
        super().__init__()
        self.latent_dim = latent_dim
        self.beta = beta

        # Encoder backbone (condiviso tra μ e log σ²)
        enc = []
        prev = input_dim
        for h in hidden:
            enc += [nn.Linear(prev, h), nn.LayerNorm(h), nn.GELU()]
            prev = h
        self.encoder_backbone = nn.Sequential(*enc)
        self.fc_mu     = nn.Linear(prev, latent_dim)
        self.fc_logvar = nn.Linear(prev, latent_dim)

        # Decoder
        dec = []
        prev = latent_dim
        for h in reversed(hidden):
            dec += [nn.Linear(prev, h), nn.LayerNorm(h), nn.GELU()]
            prev = h
        dec += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*dec)

    def encode(self, x):
        h = self.encoder_backbone(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        if self.training:
            return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return mu

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

    def elbo_loss(self, x, reco, mu, logvar):
        recon = F.mse_loss(reco, x, reduction='mean')
        kl    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        return recon + self.beta * kl, recon, kl

    def anomaly_score(self, x, n_samples=10):
        self.eval()
        with torch.no_grad():
            mu, logvar = self.encode(x)
            scores = []
            for _ in range(n_samples):
                z    = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
                reco = self.decoder(z)
                scores.append(F.mse_loss(reco, x, reduction='none').mean(dim=1))
        return torch.stack(scores).mean(dim=0)


model_vae = MLP_VAE(
    input_dim  = N_EMB,
    latent_dim = LATENT_DIM,
    hidden     = [64, 32],
    beta       = 1.0,
).to(device)

n_params_vae = sum(p.numel() for p in model_vae.parameters() if p.requires_grad)
print(model_vae)
print(f'\nParametri VAE: {n_params_vae:,}')


In [ ]:
def train_vae(model, loader_train, loader_val_qcd, n_epochs=200, lr=3e-4, patience=20, verbose=True):
    """Training loop VAE con ELBO = recon + beta*KL."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8)
    history = {'train': [], 'val': [], 'recon': [], 'kl': []}
    best_val, best_state, no_improve = float('inf'), None, 0

    for epoch in range(1, n_epochs + 1):
        model.train()
        tot_loss = tot_recon = tot_kl = 0.
        for (xb,) in loader_train:
            xb = xb.to(device)
            optimizer.zero_grad()
            reco, mu, logvar = model(xb)
            loss, recon, kl  = model.elbo_loss(xb, reco, mu, logvar)
            loss.backward()
            optimizer.step()
            n = len(xb)
            tot_loss  += loss.item()  * n
            tot_recon += recon.item() * n
            tot_kl    += kl.item()    * n
        N = len(loader_train.dataset)
        history['train'].append(tot_loss  / N)
        history['recon'].append(tot_recon / N)
        history['kl'].append(tot_kl    / N)

        model.eval()
        tot = 0.
        with torch.no_grad():
            for (xb,) in loader_val_qcd:
                xb = xb.to(device)
                reco, mu, logvar = model(xb)
                loss, _, _ = model.elbo_loss(xb, reco, mu, logvar)
                tot += loss.item() * len(xb)
        val_loss = tot / len(loader_val_qcd.dataset)
        history['val'].append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if verbose and (epoch % 20 == 0 or epoch == 1):
            print(f'Epoch {epoch:4d}/{n_epochs} | ELBO={history["train"][-1]:.5f} '
                  f'(recon={history["recon"][-1]:.5f} kl={history["kl"][-1]:.5f}) | val={val_loss:.5f}')

        if no_improve >= patience:
            print(f'Early stopping a epoch {epoch}.')
            break

    model.load_state_dict(best_state)
    print(f'Best val ELBO: {best_val:.5f}')
    return history


print('=== Training VAE [no-disp] ===')
t0 = time.time()
history_vae = train_vae(model_vae, loader_train, loader_val_qcd, n_epochs=200, lr=3e-4, patience=20)
print(f'Tempo totale: {time.time()-t0:.1f}s')


In [ ]:
# ── Loss curves VAE ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, len(history_vae['train']) + 1)
axes[0].plot(ep, history_vae['train'], label='ELBO train')
axes[0].plot(ep, history_vae['val'],   label='ELBO val', linestyle='--')
axes[0].set_title('VAE [no-disp] — ELBO'); axes[0].legend()
axes[1].plot(ep, history_vae['recon'], label='Recon')
axes[1].plot(ep, history_vae['kl'],    label='KL × β', linestyle='--')
axes[1].set_title('VAE [no-disp] — Recon vs KL'); axes[1].legend()
for ax in axes: ax.set_xlabel('Epoch')
fig.tight_layout()
fig.savefig(OUT_DIR / 'vae_loss.png', dpi=120)
plt.show()


In [ ]:
# ── Anomaly scores VAE ────────────────────────────────────────────────────────
Xte_t = torch.from_numpy(X_test).to(device)
scores_vae = model_vae.anomaly_score(Xte_t, n_samples=10).cpu().numpy()
s_qcd_vae  = scores_vae[y_test == 0]
s_ej_vae   = scores_vae[y_test == 1]
auc_vae    = roc_auc_score(y_test, scores_vae)


## 6. Confronto AE vs VAE [no-disp]

In [ ]:
print('=== Confronto AE vs VAE [no-disp] ===')
print(f'{"Modello":12s}  {"AUC":>6s}  {"QCD mean":>10s}  {"EJ mean":>10s}')
print('-' * 48)
print(f'{"AE  no-disp":12s}  {auc_ae:.4f}  {s_qcd_ae.mean():10.4f}  {s_ej_ae.mean():10.4f}')
print(f'{"VAE no-disp":12s}  {auc_vae:.4f}  {s_qcd_vae.mean():10.4f}  {s_ej_vae.mean():10.4f}')
print()
print('Per confronto con 06_autoencoder (tutte le feature):')
print('  AUC AE  (all feat) ≈ 0.9984')


In [ ]:
# ── Score distributions ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (sq, se, title) in zip(axes, [
    (s_qcd_ae,  s_ej_ae,  f'AE  [no-disp]  (AUC={auc_ae:.4f})'),
    (s_qcd_vae, s_ej_vae, f'VAE [no-disp]  (AUC={auc_vae:.4f})'),
]):
    lo = np.percentile(np.concatenate([sq, se]), 0.5)
    hi = np.percentile(np.concatenate([sq, se]), 99.5)
    bins = np.linspace(lo, hi, 80)
    ax.hist(sq, bins=bins, density=True, alpha=0.6, label='QCD', color='steelblue')
    ax.hist(se, bins=bins, density=True, alpha=0.6, label='EJ',  color='tomato')
    ax.set_yscale('log')
    ax.set_xlabel('Anomaly score (MSE recon)')
    ax.set_title(title)
    ax.legend()

fig.tight_layout()
fig.savefig(OUT_DIR / 'ae_vs_vae_scores.png', dpi=120)
plt.show()


In [ ]:
# ── ROC comparativa ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
for scores, label, color in [
    (scores_ae,  f'AE  no-disp  AUC={auc_ae:.4f}',  'steelblue'),
    (scores_vae, f'VAE no-disp  AUC={auc_vae:.4f}', 'tomato'),
]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    ax.plot(fpr, tpr, label=label, color=color)
ax.plot([0,1],[0,1],'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC — AE vs VAE [no-disp]')
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / 'roc_ae_vs_vae.png', dpi=120)
plt.show()


## 7. Salvataggio

In [ ]:
import json

# Pesi dei modelli
torch.save(model_ae.state_dict(),  OUT_DIR / 'ae_weights.pt')
torch.save(model_vae.state_dict(), OUT_DIR / 'vae_weights.pt')

# Config
cfg = {
    'input_dim':        N_EMB,
    'input_dim_orig':   86,
    'removed_features': sorted(list(REMOVE_TRACK | REMOVE_JET)),
    'keep_dims':        FEAT_MASK.tolist(),
    'latent_dim':       LATENT_DIM,
    'hidden':           [64, 32],
    'ae_dropout':       0.05,
    'vae_beta':         float(model_vae.beta),
    'seed':             SEED,
}
with open(OUT_DIR / 'model_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

# Scores test set
np.savez(
    OUT_DIR / 'test_scores.npz',
    scores_ae  = scores_ae,
    scores_vae = scores_vae,
    labels     = y_test,
    auc_ae     = np.array(auc_ae),
    auc_vae    = np.array(auc_vae),
    feat_mask  = FEAT_MASK,
)

print('Salvati in:', OUT_DIR)
for p in [OUT_DIR/'ae_weights.pt', OUT_DIR/'vae_weights.pt',
          OUT_DIR/'model_config.json', OUT_DIR/'test_scores.npz']:
    print(f'  {p}')
print(f'\nAUC AE  [no-disp] = {auc_ae:.4f}')
print(f'AUC VAE [no-disp] = {auc_vae:.4f}')
print('\n✓ Task 2 (no-disp) completato. Confronta questi AUC con il 06.')
